<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.3

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate

**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for this lecture:** the
[opencode documentation](https://opencode.ai/docs/) and Brown CCV's
[Oscar documentation](https://docs.ccv.brown.edu/oscar/).

**How:** This one is yours to run: a coding agent on your own machine, driven by a model on
an Oscar GPU.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

## Outline for today

* Three things to have before the script will run
* `agentsoscar.sh`, and what it does on your behalf
* Sessions, and the commands that matter
* Traps we already fell into

**Reading material**: `agentsoscar.sh` in this folder, the
[opencode docs](https://opencode.ai/docs/), and
[CCV's Oscar docs](https://docs.ccv.brown.edu/oscar/).

The goal of this lecture is independence. By the end they should be able to run a coding agent
against a model they control, on hardware they have access to, and debug it when it fails.

They will not run opencode directly, and they will not use a hosted provider. One script, one
model, one cluster.

## Three things, in the order they fail

1. `opencode` on your own machine &nbsp;([install](https://opencode.ai/install))
2. An SSH entry named `oscar-campus` &nbsp;([CCV: SSH configuration file](https://docs.ccv.brown.edu/oscar/connecting-to-oscar/ssh/ssh-configuration-file))
3. An Oscar account &nbsp;([CCV docs](https://docs.ccv.brown.edu/oscar/)), and the Brown VPN off campus

```bash
curl -fsSL https://opencode.ai/install | bash     # Installs opencode
mkdir -p ~/.local/bin && cp agentsoscar.sh ~/.local/bin/ # Makes it available everywhere on your computer, assumes macOS or Linux
chmod +x ~/.local/bin/agentsoscar.sh              # then: agentsoscar.sh
```

The script checks them in this order and each failure looks different, which is why the order
is worth stating.

A missing agent now stops the script immediately, before any allocation, and prints the install
command. That check did not exist until this lecture: previously a student without opencode
allocated a GPU, waited out an 18 GB download, got a tunnel, and only then hit command not found.

No oscar-campus entry and it prints the CCV configuration page and exits before touching Slurm.
The name has to match exactly, because that string is REMOTE_HOST. Off campus without the VPN it
waits 60 seconds, then names the VPN, a missed Duo push, and running ssh oscar-campus by hand.

If agentsoscar.sh --help says command not found after the copy, ~/.local/bin is not on your PATH.
Add export PATH="$HOME/.local/bin:$PATH" to ~/.zshrc and open a new terminal. Or skip the copy and
run ./agentsoscar.sh in place.

The script ships next to this notebook, and its header repeats all of this.

## The model you get

<p align="center"><img src="../figures/opencode_tui.png" width="72%"></p>

`qwen3.8:27b` at 256k context, served from an Oscar GPU. 27B dense, 18 GB of weights,
Apache 2.0.

The bottom line of the prompt box is the proof the tunnel works: Build, then the model name,
then Ollama as the provider. If that line says anything else, the script did not finish wiring up.

Dense means every parameter is active on every token, unlike the previous default qwen3.6:35b,
which is a mixture of experts with roughly 3B active. The 262,144-token window in the script
matches this model's native context exactly.

First run downloads 18 GB into /oscar/scratch/$USER/ollama-models and keeps it, so later runs
start fast. Pass --purge-cache to delete it on exit. Change either default with -m and --ctx.

Two keybindings are on screen and worth pointing at: tab switches agent, ctrl+p opens the command
palette. Slash commands work too, but the palette is how you discover them.

## Inside the script

<p align="center"><img src="../figures/opencode_oscar_tunnel.png" width="88%"></p>

It runs the **agent on your laptop** and the **model on Oscar**, joined by an SSH tunnel.

```
salloc -p gpu --gres=gpu:l40s:1 ... srun    # a GPU node, via srun
  -> download ollama to /oscar/scratch     # not the module: see notes
  -> ollama serve on a per-run port
  -> ollama create <model>-256k            # num_ctx baked in
ssh -L 127.0.0.1:PORT:<node-ip>:PORT       # the tunnel
  -> write ~/.config/opencode/opencode.json
  -> launch opencode
```

Two details worth knowing because they explain most failures.

The workload goes through srun inside the allocation, not bare salloc. `salloc <resources> <cmd>`
runs the command where you invoked it, which is the login node. The script also makes the job
write its own hostname to a file and refuses to continue if that does not match the allocated
node. That check exists because the alternative once put a model server on a shared login node.

Oscar's ollama module is 0.21.0 and crashed this class of model at 128k context with a CUDA
illegal-memory-access at 28 GiB, which is not a memory limit. The script downloads v0.32.12 into
scratch instead.

When something fails, the log is on Oscar:
ssh oscar-campus cat /oscar/scratch/$USER/opencode-runs/<run-id>/launch.log

## The tools run on your laptop

<p align="center"><img src="../figures/opencode_tui_2.png" width="66%"></p>

The model is on Oscar. Everything it *does* happens here.

**It did not ask.**

This one screenshot carries the architecture. The model reasoning on a GPU node correctly
concluded "On macOS (darwin)" and reached for top and ps, because the tools execute on the
machine you launched from. The model is remote; the filesystem, the shell and the risk are local.

Say the second fragment plainly: it ran a shell command on a laptop without asking first. There
is a /permissions command, but do not rely on it, and do not assume you will be prompted. If you
want a gate, configure it before you need it.

Two other things on screen. Thought: 1.5s is the reasoning trace, shown by default. And 11.1K in
the status bar is context consumed, which is the number that makes /compact necessary rather than
optional.

GPU contention, previously its own slide, is now in the notes: the shared pool is usually full,
the script retries across l40s, a6000 and a40, and you can check availability yourself with
ssh oscar-campus sinfo -p gpu -N -o '%N %t %G'. The partition is gpu, not gpu-debug, whose QOS
caps jobs at one hour.

## A session

<p align="center"><img src="../figures/opencode_tui_3.png" width="70%"></p>

Everything opencode remembers about one line of work, stored on disk in
`~/.local/share/opencode`. Quitting does not end it.

`/sessions` to browse, `/resume` to reopen, `/new` for a clean one, `/compact` to summarise a
long one.

The picker is the argument: sessions are objects, not a mood. They are titled automatically from
their own content, grouped by day, searchable, and they persist. The oldest one on this screen is
from March.

The distinction students get wrong: quitting the agent does not end the session, and launching the
agent does not continue one. The wrapper script always starts you in a fresh session, so reach for
/resume yourself.

In the picker, ctrl+f pins, ctrl+d deletes, ctrl+r renames. From outside the TUI,
opencode session list and opencode session delete <id>.

/compact replaces the history with a summary and keeps going. Do it at a natural boundary, not
mid-edit, and watch the context counter rather than waiting for the model to start forgetting the
beginning of its own task.

## The commands worth knowing

`ctrl+p` opens the palette. `tab` switches agent.

| | |
|---|---|
| `/init` | write an `AGENTS.md` for this project |
| `/agent` | switch agent |
| `/tools` | what it can call |
| `/skills` | reusable instructions, loaded on demand |
| `/mcp` | connect an external tool server |
| `/undo` `/redo` | step its edits back and forward |

AGENTS.md is the project's standing instructions, the same idea as CLAUDE.md, which opencode
also reads. /init drafts one by reading the repository. Everything 19.1 said about personas
applies: a system prompt plus a tool list is what an agent is, and these are where you set both.

Skills are directories, .opencode/skills/<name>/SKILL.md in a project or
~/.config/opencode/skill/ globally, loaded when relevant rather than always, so they cost nothing
until used.

Custom agents live in .opencode/agents/, custom slash commands in .opencode/commands/, and
opencode agent create scaffolds one.

/permissions exists but has not done anything useful for us in this setup, which is why it is not
on the slide. Treat the agent as unrestricted until you have configured otherwise.

## Traps

**The script overwrites `~/.config/opencode/opencode.json` every run.** Keep your own settings in
the project's `.opencode/` instead.

**The port changes every run.** Typing `opencode` by hand after the script exits points it at a
tunnel that is gone.

**It will not ask before acting.** Assume any command it decides to run, runs. Never add
`--auto` on top of that.

The config file is generated from scratch on every launch, so an MCP server or a keybinding you
add there is gone the next time. Project-level .opencode/ survives.

The port is 20000 plus the process id modulo 10000, so it is different every run by design: the
default 11434 collides, because GPU nodes are multi-tenant and someone else's job had already
bound it.

Two more from the same file, both worth saying out loud because they cost real time. Ollama sizes
its KV cache as num_ctx multiplied by OLLAMA_NUM_PARALLEL, and that variable has been reported
ignored in favour of a hardcoded 4, which at this context would exhaust a 48 GB card before the
weights loaded; the script pins it to 1. And the compute node reports its own IP to a file rather
than letting the login node resolve it, because self-resolution on a multi-homed node does not
reliably agree.

If you ever cancel a job by hand, cancel it by the name you created. Never scancel everything you
own.

## The backend choice

Both are supported. `--backend claude` exists.

Ollama's Anthropic-compatible endpoint rejects any conversation whose history contains a system
message that is not the first one. Claude Code sends many, scattered. It failed **10 out of 10**
attempts where opencode succeeded immediately on the same server and model.

The reason is protocol, not quality. opencode talks to /v1/chat/completions, the
OpenAI-compatible endpoint, which has no such restriction. Claude Code talks to /v1/messages,
where the model's own chat template enforces system-message ordering.

This was established by capturing the actual requests through a local logging proxy, not inferred
from documentation. --backend claude is still fine for a short one-off question, and remains in
the script in case Ollama's validator changes.

The transferable lesson: when an agent and a server disagree, the first thing to check is which
endpoint is being used and what it accepts.

## Summary

* The agent runs locally. The model runs on a GPU you allocated. A tunnel joins them.
* One script does all of it, and tears it down when you exit.
* A session is a stored object, and `/compact` is what keeps a long one usable.
* When it fails, read `launch.log` on Oscar before changing anything.

## Your turn

1. Get `oscar-campus` working by hand: `ssh oscar-campus` should just work.
   [CCV: SSH configuration file](https://docs.ccv.brown.edu/oscar/connecting-to-oscar/ssh/ssh-configuration-file)
2. Run `agentsoscar.sh` and wait out the first download.
3. Inside, run `/init`, then read the `AGENTS.md` it wrote and correct it.
4. Build one agent with `opencode agent create` and give it only the tools it needs.
   [opencode docs](https://opencode.ai/docs/)

### See you next class

Have fun!